# Projeto Fictus | Análise Financeira — Bloco 1: Perfil de Receita e Pagamento

---

## Pergunta Central do Bloco
> **O modelo de pagamentos da empresa-alvo é compatível com crescimento sustentável — ou gera pressão crescente sobre o caixa e a necessidade de capital de giro do comprador?**

---

## Contexto do Bloco

A Análise de Logística mapeou o modelo operacional de entrega da empresa-alvo e seus limites de escalabilidade. A Análise de Finanças completa a due diligence examinando a estrutura financeira da operação.

Entender como o dinheiro entra é tão crítico quanto entender como ele é gerado. Uma empresa pode crescer em receita e, simultaneamente, deteriorar sua posição de caixa — se o crescimento for financiado por terceiros via parcelamento. Este bloco mapeia a arquitetura de recebimento da empresa-alvo para revelar quem, de fato, está financiando o crescimento.

**Este bloco investiga:**
1. Mix de Pagamento: à Vista vs. Parcelado
2. Capital em Aberto: Quem Financia o Crescimento?
3. Prazo Médio de Recebimento por Modalidade
4. Descasamento Temporal: Pico de Venda vs. Pico de Caixa
5. Variabilidade do Recebimento

---

## Nota Metodológica
O dataset Olist registra o tipo de pagamento e o número de parcelas por pedido, mas não o fluxo de caixa real.
As variáveis de PMR e capital em aberto são **estimativas derivadas com premissas declaradas no ETL**.
Toda a análise é baseada nesses proxies — não em dados bancários reais.

---


## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_FIN     = BASE_DIR / "data" / "finance"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"
COR_CAIXA    = "#148F77"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    elif abs(x) >= 1_000:   return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def fmt_dias(x, pos=None): return f"{x:.0f}d"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento dos Dados

In [ ]:
def ler(f, **kw):
    df = pd.read_csv(DIR_FIN / f, low_memory=False, **kw)
    df.columns = df.columns.str.strip()
    return df

fin_fato   = ler("fin_fato.csv")
fin_mensal = ler("fin_mensal.csv")
fin_trim   = ler("fin_trimestral.csv")
fin_pag    = ler("fin_pagamento.csv")
fin_reg    = ler("fin_regional.csv")
fin_faixa  = ler("fin_faixa.csv")

for col in ["preco", "valor_frete", "pmr_ajustado", "spread_intermediario",
            "capital_em_aberto", "anomalia_pagamento", "numero_parcelas"]:
    if col in fin_fato.columns:
        fin_fato[col] = pd.to_numeric(fin_fato[col], errors="coerce")

periodos_ord = sorted(fin_fato["periodo"].dropna().unique())
print(f"Dados: {len(fin_fato):,} registros | {periodos_ord[0]} a {periodos_ord[-1]}")
print(f"Modalidades: {fin_fato['tipo_pagamento'].unique().tolist()}")


---

## Análise 1 — Mix de Pagamento: à Vista vs. Parcelado

> *"A estrutura de recebimento é o principal determinante da autonomia financeira de um ativo. O mix entre pagamentos à vista e parcelados revela o custo financeiro implícito no modelo de vendas e a dependência de intermediários financeiros para a sustentação do faturamento. Através da aplicação do Princípio de Pareto, esta análise identifica as modalidades que concentram o volume transacional e quantifica a parcela da receita que permanece retida sob a forma de recebíveis de longo prazo."*

**Framework:** Pareto — decomposição do mix de pagamento  
**Entrega:** Distribuição da receita por modalidade e faixa de parcelamento, com evolução temporal

**Como este script responde à pergunta:**
> O script calcula a participação de cada modalidade de pagamento na receita total e agrupa as vendas por faixa de parcelamento. Três painéis respondem à pergunta:
> 1. **Participação por modalidade:** Barras com o percentual de cada forma de pagamento na receita total — anotado diretamente em cada barra. Revela de imediato qual modalidade domina e, portanto, qual intermediário mais captura spread.
> 2. **Curva de Pareto acumulada:** Barras de participação com linha de percentual acumulado sobreposta e marcação do limiar de 80%. Mostra em quantas modalidades se concentra a maior parte da receita comprometida com terceiros.
> 3. **Receita por faixa de parcelamento:** Barras por faixa (à vista, 2x–3x, 4x–6x, 7x–12x, acima de 12x), com o pagamento à vista destacado em verde e o parcelado em azul. A distribuição entre as faixas define o tamanho do capital que ainda não entrou no caixa.

**Análise do Resultado:**
A distribuição das faixas de parcelamento atua como um indicador da necessidade de capital de giro. Uma concentração elevada em faixas superiores (ex: 7x a 12x) indica que o crescimento da receita contábil não é acompanhado pela entrada imediata de caixa. Para um investidor, este diagnóstico define o montante de capital que precisará ser aportado para sustentar o fluxo operacional até que os recebíveis sejam liquidados, impactando diretamente o valuation e a estrutura do deal.

In [ ]:
# ─── Mix de pagamento: participação na receita ────────────────────────────────
receita_total = fin_fato["preco"].sum()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Bloco 1 — Mix de Pagamento e Estrutura de Recebimento", fontsize=13, fontweight="bold")

# Painel 1: Participação por modalidade
cores_modal = [COR_RECEITA, COR_ALERTA, COR_DESTAQUE, COR_MARGEM, COR_NEUTRO]
bars1 = axes[0].bar(fin_pag["tipo_pagamento"], fin_pag["pct_receita"],
                    color=cores_modal[:len(fin_pag)], alpha=0.85)
axes[0].set_title("Participação na Receita por Modalidade", fontsize=10)
axes[0].set_ylabel("% da Receita Total")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].tick_params(axis="x", rotation=20)
for bar, val in zip(bars1, fin_pag["pct_receita"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Painel 2: Curva de Pareto acumulada
axes[1].bar(fin_pag["tipo_pagamento"], fin_pag["pct_receita"],
            color=COR_RECEITA, alpha=0.5, label="Participação")
ax2 = axes[1].twinx()
ax2.plot(fin_pag["tipo_pagamento"], fin_pag["pct_acum"],
         color=COR_ALERTA, marker="o", linewidth=2, label="% Acumulado")
ax2.axhline(80, color=COR_ALERTA, linestyle="--", alpha=0.4, linewidth=1)
ax2.set_ylim(0, 110)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Curva de Pareto — Concentração de Receita", fontsize=10)
axes[1].tick_params(axis="x", rotation=20)

# Painel 3: Participação por faixa de parcelamento
ORDEM_FAIXAS = ["a_vista", "2x_3x", "4x_6x", "7x_12x", "13x_mais"]
fin_faixa_ord = fin_faixa.set_index("faixa_parcelamento").reindex(
    [f for f in ORDEM_FAIXAS if f in fin_faixa["faixa_parcelamento"].values]
).reset_index()
bars3 = axes[2].bar(fin_faixa_ord["faixa_parcelamento"], fin_faixa_ord["pct_receita"],
                    color=[COR_MARGEM if f == "a_vista" else COR_RECEITA for f in fin_faixa_ord["faixa_parcelamento"]],
                    alpha=0.85)
axes[2].set_title("Receita por Faixa de Parcelamento", fontsize=10)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[2].tick_params(axis="x", rotation=20)
for bar, val in zip(bars3, fin_faixa_ord["pct_receita"]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
salvar(fig, "01_mix_pagamento")
plt.show()

print("\n── Mix de Pagamento ──────────────────────")
print(fin_pag[["tipo_pagamento", "pct_receita", "pct_acum", "pmr_medio", "pct_spread"]].to_string(index=False))


---

## Análise 2 — Capital em Aberto: Quem Financia o Crescimento?

> *"Receita reconhecida e caixa disponível são números diferentes em qualquer operação com parcelamento relevante. Esta análise compara as duas séries ao longo do tempo para revelar o gap estrutural entre o que a empresa vende e o que efetivamente recebe — e quanto desse gap cresce junto com o volume. O spread capturado por intermediários é o custo visível desse modelo; o capital em aberto é o custo invisível que pressiona o caixa do comprador."*

**Framework:** Controle de processo — Engenharia de Produção aplicada ao fluxo de caixa  
**Entrega:** Série temporal de capital em aberto versus receita reconhecida, com razão de financiamento externo

**Como este script responde à pergunta:**
> O script estima mensalmente o capital vendido mas ainda não recebido, com base no mix de parcelamento e nas taxas de desconto declaradas no ETL. Quatro painéis respondem à pergunta:
> 1. **Receita vs. Capital em Aberto (série temporal):** Linha de receita (eixo esquerdo) e linha tracejada de capital em aberto (eixo direito) no mesmo gráfico. Se o capital em aberto cresce mais rápido que a receita, o crescimento está sendo financiado por terceiros.
> 2. **Capital em aberto como % da receita mensal:** Barras coloridas — vermelho quando acima da média, azul quando abaixo — com linha de referência da média. Meses vermelhos são meses em que a dependência de intermediários foi maior.
> 3. **Spread estimado capturado por intermediários (R$ mil):** Barras mensais do spread total em reais — o custo de terceirizar o recebimento mês a mês.
> 4. **Spread como % da receita:** Linha com marcadores e linha tracejada de média. Se a linha sobe, os intermediários estão capturando proporcionalmente mais do crescimento.

**Análise do Resultado:**
Se a taxa de crescimento do capital em aberto superar a taxa de crescimento da receita, o ativo apresenta um risco de sustentabilidade financeira. O aumento do spread como percentual da receita sinaliza que o custo de intermediação está crescendo proporcionalmente ao volume — a operação escala, mas os intermediários capturam uma fatia proporcionalmente maior desse crescimento. Este dado orienta o comprador sobre a eficiência financeira real da operação e a urgência de renegociar condições com adquirentes antes do fechamento.


In [ ]:
# ─── Capital em aberto vs. receita: quem financia? ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Bloco 1 — Capital em Aberto e Financiamento do Crescimento", fontsize=13, fontweight="bold")

# Painel 1: Receita vs. Capital em Aberto (série temporal)
axes[0, 0].fill_between(range(len(fin_mensal)), fin_mensal["receita_total"] / 1000,
                         alpha=0.3, color=COR_RECEITA, label="Receita")
axes[0, 0].plot(range(len(fin_mensal)), fin_mensal["receita_total"] / 1000,
                color=COR_RECEITA, linewidth=2)
ax_cap = axes[0, 0].twinx()
ax_cap.plot(range(len(fin_mensal)), fin_mensal["capital_aberto"] / 1000,
            color=COR_ALERTA, linewidth=2, linestyle="--", marker="o", markersize=4,
            label="Capital em Aberto")
axes[0, 0].set_xticks(range(len(fin_mensal)))
axes[0, 0].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[0, 0].set_title("Receita vs. Capital em Aberto (R$ mil)", fontsize=10)
axes[0, 0].set_ylabel("Receita (R$ mil)", color=COR_RECEITA)
ax_cap.set_ylabel("Capital em Aberto (R$ mil)", color=COR_ALERTA)
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
ax_cap.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))

# Painel 2: Razão de financiamento externo
razao = fin_mensal["razao_financiamento"].fillna(0)
axes[0, 1].bar(range(len(fin_mensal)), razao * 100, color=[
    COR_ALERTA if r > razao.mean() else COR_RECEITA for r in razao
], alpha=0.8)
axes[0, 1].axhline(razao.mean() * 100, color=COR_NEUTRO, linewidth=1.5,
                   linestyle="--", label=f"Média: {razao.mean()*100:.1f}%")
axes[0, 1].set_xticks(range(len(fin_mensal)))
axes[0, 1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[0, 1].set_title("Capital em Aberto como % da Receita Mensal", fontsize=10)
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0, 1].legend(fontsize=8)

# Painel 3: Spread capturado por intermediários
axes[1, 0].bar(range(len(fin_mensal)), fin_mensal["spread_total"] / 1000,
               color=COR_DESTAQUE, alpha=0.8)
axes[1, 0].set_xticks(range(len(fin_mensal)))
axes[1, 0].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1, 0].set_title("Spread Estimado Capturado por Intermediários (R$ mil)", fontsize=10)
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1, 0].set_ylabel("Spread (R$ mil)")

# Painel 4: % Spread sobre receita
axes[1, 1].plot(range(len(fin_mensal)), fin_mensal["pct_spread_receita"],
                color=COR_DESTAQUE, linewidth=2, marker="o", markersize=5)
axes[1, 1].axhline(fin_mensal["pct_spread_receita"].mean(), color=COR_NEUTRO,
                   linestyle="--", linewidth=1.5, label=f"Média: {fin_mensal['pct_spread_receita'].mean():.1f}%")
axes[1, 1].set_xticks(range(len(fin_mensal)))
axes[1, 1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1, 1].set_title("Spread Intermediários como % da Receita", fontsize=10)
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "01_capital_aberto")
plt.show()

print("\n── Resumo Financeiro do Período ──────────────────────")
print(f"  Spread total capturado por intermediários : R$ {fin_fato['spread_intermediario'].sum():,.0f}")
print(f"  Capital em aberto total estimado          : R$ {fin_fato['capital_em_aberto'].sum():,.0f}")
print(f"  Spread médio como % da receita            : {(fin_fato['spread_intermediario'].sum() / fin_fato['preco'].sum() * 100):.2f}%")
print(f"  Razão média de financiamento externo      : {(fin_mensal['razao_financiamento'].mean() * 100):.1f}%")


---

## Análise 3 — Prazo Médio de Recebimento por Modalidade

> *"O Prazo Médio de Recebimento (PMR) é o termômetro da liquidez de curto prazo. Esta análise decompõe o PMR por modalidade para rastrear a tendência de estabilidade ou crescimento dos prazos de liquidação. A utilização de regressão linear para calcular o slope (inclinação) da curva de PMR permite prever se o ativo está caminhando para um cenário de estrangulamento de caixa, independentemente do sucesso das vendas."*

**Framework:** PDCA — monitoramento de indicador financeiro  
**Entrega:** PMR por modalidade e por período, com análise de tendência

**Como este script responde à pergunta:**
> O script calcula o PMR por modalidade ajustado pelo número de parcelas, e rastreia sua evolução ao longo do tempo. Dois painéis respondem à pergunta:
>1. **PMR médio por modalidade (barras horizontais):** Ordenado do menor para o maior prazo, com código de cor: verde até 30 dias, laranja até 60, vermelho acima de 60. Mostra instantaneamente quais modalidades comprimem o caixa com mais intensidade.
>2. **Evolução do PMR médio ao longo do tempo:** Linha mensal com linha de tendência por regressão linear. O slope anotado na legenda é o termômetro: positivo indica que o prazo médio de recebimento está crescendo — a empresa está vendendo mais mas recebendo proporcionalmente mais tarde.

**Análise do Resultado:**
Um PMR crescente em um cenário de volume em expansão é um sinal de alerta estrutural — a empresa está vendendo mais mas recebendo proporcionalmente mais tarde, aumentando sua exposição financeira a cada ciclo de crescimento. Para o comprador, um slope positivo e significativo é um indicador direto de deterioração do modelo de recebimento: a renegociação de condições com adquirentes e intermediários precisa estar no plano dos primeiros 100 dias pós-aquisição, antes que o crescimento futuro aprofunde ainda mais o descasamento entre receita e caixa.


In [ ]:
# ─── PMR por modalidade e evolução temporal ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Bloco 1 — Prazo Médio de Recebimento (PMR)", fontsize=13, fontweight="bold")

# Painel 1: PMR por modalidade
fin_pag_ord = fin_pag.sort_values("pmr_medio", ascending=True)
cores_pmr = [COR_ALERTA if p > 60 else COR_DESTAQUE if p > 30 else COR_MARGEM
             for p in fin_pag_ord["pmr_medio"]]
bars = axes[0].barh(fin_pag_ord["tipo_pagamento"], fin_pag_ord["pmr_medio"],
                    color=cores_pmr, alpha=0.85)
axes[0].set_title("PMR Médio por Modalidade de Pagamento (dias)", fontsize=10)
axes[0].set_xlabel("Dias")
for bar, val in zip(bars, fin_pag_ord["pmr_medio"]):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f"{val:.0f}d", va="center", fontsize=9, fontweight="bold")

# Painel 2: Evolução do PMR médio mensal
axes[1].plot(range(len(fin_mensal)), fin_mensal["pmr_medio"],
             color=COR_RECEITA, linewidth=2, marker="o", markersize=5)
slope, intercept, r, p, _ = stats.linregress(range(len(fin_mensal)),
                                              fin_mensal["pmr_medio"].fillna(0))
tend_vals = [intercept + slope * i for i in range(len(fin_mensal))]
axes[1].plot(range(len(fin_mensal)), tend_vals, color=COR_ALERTA,
             linestyle="--", linewidth=1.5, alpha=0.7,
             label=f"Tendência (slope={slope:+.2f}d/mês)")
axes[1].set_xticks(range(len(fin_mensal)))
axes[1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1].set_title("Evolução do PMR Médio ao Longo do Tempo", fontsize=10)
axes[1].set_ylabel("PMR Médio (dias)")
axes[1].legend(fontsize=8)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dias))

plt.tight_layout()
salvar(fig, "01_pmr_modalidade")
plt.show()

print("\n── PMR por Modalidade ──────────────────────")
print(fin_pag[["tipo_pagamento", "pmr_medio", "pct_receita", "parcelas_medio"]].to_string(index=False))
print(f"\n  PMR médio geral (ponderado): {(fin_fato['pmr_ajustado'] * fin_fato['preco']).sum() / fin_fato['preco'].sum():.0f} dias")
print(f"  Tendência PMR (slope)       : {slope:+.2f} dias/mês — {'⚠️  crescente' if slope > 0.5 else '✅ estável'}")


---

## Análise 4 — Descasamento Temporal: Pico de Venda vs. Pico de Caixa

> *"Sazonalidade de faturamento e sazonalidade de disponibilidade de caixa raramente coincidem em modelos de varejo e marketplace. Esta análise quantifica o descasamento temporal entre a receita reconhecida pelo regime de competência e o caixa efetivamente disponível. O objetivo é determinar o pico de exposição financeira — o momento em que a operação está mais vulnerável a uma crise de liquidez — e calcular o capital de reserva mínimo necessário para atravessar os períodos de maior demanda sem recorrer a linhas de crédito externas onerosas."*

**Framework:** Análise de fluxo de processo   
**Entrega:** Gráfico de descasamento temporal entre receita reconhecida e caixa recebido por mês

**Como este script responde à pergunta:**
> O script estima o caixa efetivamente disponível em cada mês como a parcela da receita não comprometida com parcelamentos futuros, e calcula a diferença em relação à receita reconhecida. Dois painéis respondem à pergunta:
>1. **Receita reconhecida vs. Caixa estimado:** Duas áreas preenchidas sobrepostas — azul para receita, verde para caixa. O espaço entre as curvas é o descasamento: quanto maior a distância, maior o capital que entrou no resultado contábil mas ainda não chegou ao banco.
>2. **Descasamento absoluto mensal (R$ mil):** Barras com código de cor — vermelho nos meses acima da mediana, laranja nos demais — com linha de referência da mediana. O mês de maior barra é o pico de exposição, e o valor máximo multiplicado por 1,2 é a sugestão de capital de reserva mínimo.

**Análise do Resultado:**
O cálculo do descasamento máximo fornece uma métrica objetiva de segurança operacional. A distância entre as curvas de receita e caixa representa o volume de capital que permanece retido nos intermediários durante os meses de pico. Para o comprador, o valor sugerido de capital de reserva — calculado como o descasamento máximo multiplicado por 1,2 — deve ser subtraído da disponibilidade imediata ou considerado como investimento inicial obrigatório para a continuidade operacional sem recorrer a linhas de crédito externas.

In [ ]:
# ─── Descasamento temporal: receita vs. caixa disponível ─────────────────────
# Estimativa do caixa efetivo = receita × (1 - razão_financiamento)
# Representa a parcela da receita que já entrou no caixa no mês
fin_mensal["caixa_estimado"] = fin_mensal["receita_total"] * (1 - fin_mensal["razao_financiamento"].fillna(0))
fin_mensal["descasamento"]   = fin_mensal["receita_total"] - fin_mensal["caixa_estimado"]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Bloco 1 — Descasamento entre Receita Reconhecida e Caixa Disponível",
             fontsize=13, fontweight="bold")

# Painel 1: Receita vs. Caixa estimado
axes[0].fill_between(range(len(fin_mensal)), fin_mensal["receita_total"] / 1000,
                     alpha=0.3, color=COR_RECEITA)
axes[0].plot(range(len(fin_mensal)), fin_mensal["receita_total"] / 1000,
             color=COR_RECEITA, linewidth=2, label="Receita Reconhecida")
axes[0].fill_between(range(len(fin_mensal)), fin_mensal["caixa_estimado"] / 1000,
                     alpha=0.3, color=COR_MARGEM)
axes[0].plot(range(len(fin_mensal)), fin_mensal["caixa_estimado"] / 1000,
             color=COR_MARGEM, linewidth=2, linestyle="--", label="Caixa Estimado")
axes[0].set_xticks(range(len(fin_mensal)))
axes[0].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[0].set_title("Receita Reconhecida vs. Caixa Estimado (R$ mil)", fontsize=10)
axes[0].set_ylabel("R$ mil")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[0].legend(fontsize=9)

# Painel 2: Descasamento absoluto
cores_desc = [COR_ALERTA if d > fin_mensal["descasamento"].median() else COR_DESTAQUE
              for d in fin_mensal["descasamento"]]
axes[1].bar(range(len(fin_mensal)), fin_mensal["descasamento"] / 1000,
            color=cores_desc, alpha=0.85)
axes[1].axhline(fin_mensal["descasamento"].median() / 1000, color=COR_NEUTRO,
                linestyle="--", linewidth=1.5, label="Mediana")
axes[1].set_xticks(range(len(fin_mensal)))
axes[1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1].set_title("Descasamento Mensal: Receita − Caixa Imediato (R$ mil)", fontsize=10)
axes[1].set_ylabel("Descasamento (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "01_descasamento_caixa")
plt.show()

print("\n── Descasamento Temporal ──────────────────")
print(f"  Descasamento médio mensal  : R$ {fin_mensal['descasamento'].mean():,.0f}")
print(f"  Descasamento máximo (pico) : R$ {fin_mensal['descasamento'].max():,.0f}")
print(f"  Mês de maior exposição     : {fin_mensal.loc[fin_mensal['descasamento'].idxmax(), 'ano_mes']}")
print(f"  Capital de reserva sugerido: R$ {fin_mensal['descasamento'].max() * 1.2:,.0f} (descasamento máx. × 1,2)")


---

## Análise 5 — Variabilidade do Recebimento

> *"Escalabilidade pressupõe previsibilidade. Por meio do Coeficiente de Variação (CV) e de testes de correlação estatística, esta análise verifica se o aumento no volume de pedidos reduz ou amplifica a incerteza financeira. Um modelo saudável deve apresentar fluxo de caixa progressivamente mais estável à medida que o volume cresce — indicando que a estrutura de recebimento acompanha a maturidade operacional. Correlações positivas entre volume e percentual de capital em aberto indicam que a estrutura de pagamentos é um gargalo ativo para a escala"*

**Framework:** Controle Estatístico de Processo — variabilidade do recebimento  
**Entrega:** Coeficiente de variação do recebimento por período com correlação com volume

**Como este script responde à pergunta:**
> O script calcula o coeficiente de variação do recebimento e a correlação entre volume e exposição financeira. Dois painéis respondem à pergunta:
> 1. **Coeficiente de Variação (CV) — Receita vs. Caixa:** Duas barras lado a lado com o CV percentual de cada série. Quanto menor o CV, mais previsível o fluxo. Se o CV do caixa for muito maior que o da receita, o recebimento efetivo é mais imprevisível do que a venda — sinal de risco financeiro latente.
> 2. **Scatter volume de pedidos × % capital em aberto:** Cada ponto é um mês, com linha de regressão sobreposta. O coeficiente de correlação e o p-valor no título respondem diretamente: crescer em volume aumenta a dependência de intermediários ou não? Correlação positiva e significativa significa que escalar o negócio aprofunda a exposição financeira.

**Análise do Resultado:**
A disparidade entre o CV da receita e o CV do caixa revela o nível de risco latente. Se o caixa é significativamente mais volátil que a receita, o investidor herda uma operação de difícil planejamento orçamentário. Uma correlação positiva e estatisticamente significativa entre volume e percentual de capital em aberto confirma que o crescimento do ativo está aprofundando a exposição financeira — exigindo uma mudança estrutural na política de meios de pagamento para que o crescimento futuro gere valor real e não apenas volume contábil.


In [ ]:
# ─── Variabilidade do recebimento: CV e correlação com volume ─────────────────
receita_vals = fin_mensal["receita_total"].values
caixa_vals   = fin_mensal["caixa_estimado"].values

cv_receita = np.std(receita_vals) / np.mean(receita_vals)
cv_caixa   = np.std(caixa_vals) / np.mean(caixa_vals)

# Correlação entre volume de pedidos e % de capital em aberto
corr_vol_cap, p_val = stats.pearsonr(
    fin_mensal["n_pedidos"].fillna(0),
    fin_mensal["pct_capital_receita"].fillna(0)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Bloco 1 — Variabilidade e Previsibilidade do Recebimento", fontsize=13, fontweight="bold")

# Painel 1: CV da receita e do caixa
categorias_cv = ["Receita Reconhecida", "Caixa Estimado"]
valores_cv    = [cv_receita * 100, cv_caixa * 100]
cores_cv      = [COR_RECEITA, COR_MARGEM]
bars = axes[0].bar(categorias_cv, valores_cv, color=cores_cv, alpha=0.85, width=0.5)
axes[0].set_title("Coeficiente de Variação (CV)\nMenor = Mais Previsível", fontsize=10)
axes[0].set_ylabel("CV (%)", fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
for bar, val in zip(bars, valores_cv):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

# Painel 2: Correlação volume × % capital em aberto
axes[1].scatter(fin_mensal["n_pedidos"], fin_mensal["pct_capital_receita"],
                color=COR_RECEITA, alpha=0.7, s=80)
if len(fin_mensal) > 2:
    z = np.polyfit(fin_mensal["n_pedidos"].fillna(0),
                   fin_mensal["pct_capital_receita"].fillna(0), 1)
    p_fit = np.poly1d(z)
    x_line = np.linspace(fin_mensal["n_pedidos"].min(), fin_mensal["n_pedidos"].max(), 50)
    axes[1].plot(x_line, p_fit(x_line), color=COR_ALERTA, linestyle="--", linewidth=1.5)
axes[1].set_title(f"Volume de Pedidos × % Capital em Aberto\nr={corr_vol_cap:.2f} | p={p_val:.3f}",
                  fontsize=10)
axes[1].set_xlabel("Pedidos no Mês")
axes[1].set_ylabel("Capital em Aberto / Receita (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))

plt.tight_layout()
salvar(fig, "01_variabilidade_recebimento")
plt.show()

# ─── Interpretação ────────────────────────────────────────────────────────────
print("\n── Variabilidade do Recebimento ───────────────────────")
print(f"  CV da Receita Reconhecida : {cv_receita*100:.1f}%")
print(f"  CV do Caixa Estimado      : {cv_caixa*100:.1f}%")
print(f"  Correlação volume × capital: r={corr_vol_cap:.2f} | "
      f"{'crescimento aumenta exposição ⚠️' if corr_vol_cap > 0.4 else '✅ volume não amplifica exposição'}")
print(f"  p-valor                    : {p_val:.3f} ({'significativo' if p_val < 0.05 else 'não significativo'})")


In [ ]:
# ─── Métricas-chave do Bloco 1 ───────────────────────────────────────────────
pct_parcelado   = fin_fato[fin_fato["tipo_pagamento"] == "cartao_credito"]["preco"].sum() / fin_fato["preco"].sum() * 100
spread_total    = fin_fato["spread_intermediario"].sum()
capital_total   = fin_fato["capital_em_aberto"].sum()
receita_total   = fin_fato["preco"].sum()
pmr_geral       = (fin_fato["pmr_ajustado"] * fin_fato["preco"]).sum() / receita_total
desc_max        = fin_mensal["descasamento"].max()

# ─── Score do Bloco 1 ────────────────────────────────────────────────────────
# Score 3 = positivo | Score 2 = atenção | Score 1 = risco alto
_b1_parcelamento_alto = pct_parcelado > 60
_b1_pmr_longo         = pmr_geral > 45
_b1_spread_relevante  = (spread_total / receita_total) > 0.03

if _b1_parcelamento_alto and _b1_pmr_longo:
    _b1_score = 1
    _b1_sinal = "⚠️  Estrutura de recebimento pressionada — crescimento financiado por terceiros"
    _b1_cor   = COR_ALERTA
elif _b1_parcelamento_alto or _b1_pmr_longo:
    _b1_score = 2
    _b1_sinal = "⚠️  Atenção: PMR elevado e/ou alta concentração em parcelado"
    _b1_cor   = COR_DESTAQUE
else:
    _b1_score = 3
    _b1_sinal = "✅ Estrutura de recebimento saudável — mix equilibrado"
    _b1_cor   = COR_MARGEM

_b1_cond = (
    f"Reduzir prazo médio de recebimento (atual: {pmr_geral:.0f}d). "
    f"Mix de parcelamento representa fator de risco relevante para o comprador."
    if _b1_score <= 2
    else "Estrutura de recebimento saudável — baixo risco financeiro para o comprador."
)

print("=" * 60)
print("SÍNTESE — BLOCO 1: PERFIL DE RECEITA E PAGAMENTO")
print("=" * 60)
print(f"  Participação cartão crédito (parcelado) : {pct_parcelado:.1f}%")
print(f"  Spread capturado por intermediários      : R$ {spread_total:,.0f}")
print(f"  Spread como % da receita                 : {spread_total/receita_total*100:.2f}%")
print(f"  Capital em aberto estimado               : R$ {capital_total:,.0f}")
print(f"  PMR médio ponderado                      : {pmr_geral:.0f} dias")
print(f"  Descasamento máximo (pico)               : R$ {desc_max:,.0f}")
print(f"  CV da receita reconhecida                : {cv_receita*100:.1f}%")
print()
print(f"  Score do Bloco: {_b1_score}/3")
print(f"  Sinal        : {_b1_sinal}")
print()
print(f"  Condicionante: {_b1_cond}")
print("=" * 60)


---
*Próximo notebook: `02_inadimplencia_risco.ipynb` — O risco de crédito da empresa-alvo é estruturalmente controlado — ou apresenta volatilidade que representa risco relevante para o comprador?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
